In [1]:
from pathlib import Path
import sqlite3

project_root = Path.cwd()

print("Current working directory:")
print(project_root)

Current working directory:
C:\Projects\AnalyticsPortfolio\notebooks


In [2]:
from pathlib import Path
import sqlite3

notebook_dir = Path.cwd()
project_root = notebook_dir.parent

print("Notebook directory:")
print(notebook_dir)

print("\nProject root:")
print(project_root)

Notebook directory:
C:\Projects\AnalyticsPortfolio\notebooks

Project root:
C:\Projects\AnalyticsPortfolio


In [3]:
from pathlib import Path
import sqlite3

# Notebook is inside /notebooks, so project root is one level up
project_root = Path.cwd().parent

db_path = project_root / "data" / "database" / "alberta_home_insurance.db"
sql_script_path = project_root / "sql" / "01_create_tables.sql"

print("Database path:", db_path)
print("SQL script path:", sql_script_path)

# Read SQL script
sql_script = sql_script_path.read_text(encoding="utf-8")

# Create database and execute table creation script
conn = sqlite3.connect(db_path)

try:
    conn.executescript(sql_script)
    conn.commit()
    print("Database and tables created successfully.")
finally:
    conn.close()

Database path: C:\Projects\AnalyticsPortfolio\data\database\alberta_home_insurance.db
SQL script path: C:\Projects\AnalyticsPortfolio\sql\01_create_tables.sql
Database and tables created successfully.


In [4]:
import sqlite3
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent
db_path = project_root / "data" / "database" / "alberta_home_insurance.db"

conn = sqlite3.connect(db_path)

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

conn.close()

tables

,name
0,dim_date
1,fact_construction_cost
2,fact_cpi
3,fact_hail_loss_event
4,sqlite_sequence


# Milestone 2.5 – Import Data

In [5]:
from pathlib import Path

project_root = Path.cwd().parent
raw_data_path = project_root / "data" / "raw"

print("Raw data folder:")
print(raw_data_path)

raw_files = sorted([p for p in raw_data_path.iterdir() if p.is_file()])

print("\nFiles found:")
for file in raw_files:
    print(f"- {file.name}")

Raw data folder:
C:\Projects\AnalyticsPortfolio\data\raw

Files found:
- 1810000401_databaseLoadingData.csv
- 1810000401_databaseLoadingData_filtered_autoCPI.csv
- 18100004_TRUNCATED_DO_NOT_USE.csv
- 18100289.csv
- alberta_catastrophe_events.csv


In [6]:
import pandas as pd
from pathlib import Path

project_root = Path("C:/Projects/AnalyticsPortfolio")
raw_data_path = project_root / "data" / "raw"

file_inventory = []

for file in sorted(raw_data_path.iterdir()):
    if file.is_file():
        file_inventory.append({
            "file_name": file.name,
            "extension": file.suffix.lower(),
            "size_kb": round(file.stat().st_size / 1024, 2)
        })

file_inventory_df = pd.DataFrame(file_inventory)
file_inventory_df

,file_name,extension,size_kb
0,1810000401_databaseLoadingData.csv,.csv,308.02
1,1810000401_databaseLoadingData_filtered_autoCP...,.csv,47.28
2,18100004_TRUNCATED_DO_NOT_USE.csv,.csv,113480.26
3,18100289.csv,.csv,44233.42
4,alberta_catastrophe_events.csv,.csv,1.48


### Raw File Preview

The original CPI file `18100004.csv` was rejected because it was truncated and did not cover the full project period.

The final CPI source used for the warehouse is `1810000401_databaseLoadingData.csv`, downloaded from Statistics Canada using the database-loading format.

In [7]:
files = {
    "cpi": raw_data_path / "1810000401_databaseLoadingData.csv",
    "construction": raw_data_path / "18100289.csv",
    "events": raw_data_path / "alberta_catastrophe_events.csv"
}

# Try corrected read settings
cpi_preview = pd.read_csv(
    files["cpi"],
    encoding="utf-8-sig",
    nrows=5
)

construction_preview = pd.read_csv(
    files["construction"],
    sep="\t",
    encoding="latin1",
    nrows=5
)

events_preview = pd.read_csv(
    files["events"],
    encoding="latin1",
    nrows=5
)

print("=" * 80)
print("CPI preview")
print("=" * 80)
display(cpi_preview)

print("=" * 80)
print("Construction preview")
print("=" * 80)
display(construction_preview)

print("=" * 80)
print("Events preview")
print("=" * 80)
display(events_preview)

CPI preview


,REF_DATE,GEO,DGUID,Products and product groups,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2016-01,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,126.8,NaN,NaN,NaN,1
1,2016-02,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,127.1,NaN,NaN,NaN,1
2,2016-03,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,127.9,NaN,NaN,NaN,1
3,2016-04,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,128.3,NaN,NaN,NaN,1
4,2016-05,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,128.8,NaN,NaN,NaN,1


Construction preview


,Unnamed: 0,GEO,DGUID,Type of building,Division,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,1981-01,Fifteen census metropolitan area composite,NaN,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617908220,1.10.1,23.5,NaN,NaN,NaN,1
1,1981-01,"Halifax, Nova Scotia",2021S0503205,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617909756,5.10.1,32.4,NaN,NaN,NaN,1
2,1981-01,"Montréal, Quebec",2021S0503462,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617911676,10.10.1,24.6,NaN,NaN,NaN,1
3,1981-01,"OttawaGatineau, Ontario part, Ontario/Quebec",2021S050535505,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617912444,12.10.1,17.9,NaN,NaN,NaN,1
4,1981-01,"Toronto, Ontario",2021S0503535,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617912828,13.10.1,16.2,NaN,NaN,NaN,1


Events preview


,Event ID,Event Date,Event Name,Region,Peril,Industry Loss (CAD),Estimate Type,Primary Source,Notes,Unnamed: 9,Year,Event,Latest insured loss,Confidence
0,AB20200613,6/13/2020,Calgary Hailstorm,Calgary,Hail / Convective Storm,"1,300,000,000",Final available,IBC (CatIQ),One of Canada's costliest hail events.,NaN,2020,Calgary hailstorm,~$1.21.3B,?????
1,AB20210620*,2021 (date to verify),Calgary Hailstorm,Calgary,Hail / Convective Storm,"700,000,000+",Final available,IBC (CatIQ),Mentioned by IBC as a major recent Calgary hai...,NaN,2021,Calgary hailstorm & flooding (July 2),>$700M,?????
2,AB20230715,7/15/2023,South Calgary Hailstorm,Calgary,Hail / Convective Storm,"110,000,000",Final available,IBC,"Damaged homes, vehicles and commercial property.",NaN,2023,South Calgary hailstorm,~$110M,????
3,AB20240805,8/5/2024,Calgary Hailstorm,Calgary,Hail / Convective Storm,"3,293,000,000",Latest available,PERILS / CatIQ,Record revised estimate after claims development.,NaN,2024,Calgary hailstorm,$3.293B,?????
4,AB20250713,7/13/2025,Calgary Hailstorm,Calgary,Hail / Convective Storm,"164,000,000",Latest available,IBC (CatIQ),"Initial estimate $92M, later revised to $164M.",NaN,2025 Jul,Calgary hailstorm,$164M,?????


In [8]:
import pandas as pd
from pathlib import Path

project_root = Path("C:/Projects/AnalyticsPortfolio")
raw_data_path = project_root / "data" / "raw"

csv_files = sorted(raw_data_path.glob("*.csv"))

for file in csv_files:
    print("=" * 80)
    print(file.name)
    print("=" * 80)
    
    try:
        preview = pd.read_csv(file, nrows=5)
        print("Shape preview:", preview.shape)
        print("Columns:")
        print(list(preview.columns))
        display(preview)
    except Exception as e:
        print("Could not read as standard CSV.")
        print("Error:", e)

1810000401_databaseLoadingData.csv
Shape preview: (5, 15)
Columns:
['REF_DATE', 'GEO', 'DGUID', 'Products and product groups', 'UOM', 'UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE', 'STATUS', 'SYMBOL', 'TERMINATED', 'DECIMALS']


,REF_DATE,GEO,DGUID,Products and product groups,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2016-01,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,126.8,NaN,NaN,NaN,1
1,2016-02,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,127.1,NaN,NaN,NaN,1
2,2016-03,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,127.9,NaN,NaN,NaN,1
3,2016-04,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,128.3,NaN,NaN,NaN,1
4,2016-05,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,128.8,NaN,NaN,NaN,1


1810000401_databaseLoadingData_filtered_autoCPI.csv
Shape preview: (5, 15)
Columns:
['REF_DATE', 'GEO', 'DGUID', 'Products and product groups', 'UOM', 'UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE', 'STATUS', 'SYMBOL', 'TERMINATED', 'DECIMALS']


,REF_DATE,GEO,DGUID,Products and product groups,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2016-01,Alberta,2016A000248,All-items,2002=100,17,units,0,v41692327,23.2,133.7,NaN,NaN,NaN,1
1,2016-02,Alberta,2016A000248,All-items,2002=100,17,units,0,v41692327,23.2,133.8,NaN,NaN,NaN,1
2,2016-03,Alberta,2016A000248,All-items,2002=100,17,units,0,v41692327,23.2,135.0,NaN,NaN,NaN,1
3,2016-04,Alberta,2016A000248,All-items,2002=100,17,units,0,v41692327,23.2,135.1,NaN,NaN,NaN,1
4,2016-05,Alberta,2016A000248,All-items,2002=100,17,units,0,v41692327,23.2,135.6,NaN,NaN,NaN,1


18100004_TRUNCATED_DO_NOT_USE.csv
Shape preview: (5, 1)
Columns:
['REF_DATE\tGEO\tDGUID\tProducts and product groups\tUOM\tUOM_ID\tSCALAR_FACTOR\tSCALAR_ID\tVECTOR\tCOORDINATE\tVALUE\tSTATUS\tSYMBOL\tTERMINATED\tDECIMALS']


,REF_DATE\tGEO\tDGUID\tProducts and product groups\tUOM\tUOM_ID\tSCALAR_FACTOR\tSCALAR_ID\tVECTOR\tCOORDINATE\tVALUE\tSTATUS\tSYMBOL\tTERMINATED\tDECIMALS
0,1914-01\tCanada\t2016A000011124\tAll-items\t20...
1,1914-01\tCanada\t2016A000011124\tAll-items (19...
2,1914-01\tCanada\t2016A000011124\tGoods and ser...
3,1914-02\tCanada\t2016A000011124\tAll-items\t20...
4,1914-02\tCanada\t2016A000011124\tAll-items (19...


18100289.csv
Could not read as standard CSV.
Error: 'utf-8' codec can't decode byte 0xe9 in position 435: invalid continuation byte
alberta_catastrophe_events.csv
Could not read as standard CSV.
Error: 'utf-8' codec can't decode byte 0x96 in position 326: invalid start byte


In [9]:
construction_preview = pd.read_csv(
    files["construction"],
    sep="\t",
    encoding="cp1252",
    nrows=5
)

events_preview = pd.read_csv(
    files["events"],
    encoding="cp1252",
    nrows=5
)

display(construction_preview)
display(events_preview)

,Unnamed: 0,GEO,DGUID,Type of building,Division,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,1981-01,Fifteen census metropolitan area composite,NaN,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617908220,1.10.1,23.5,NaN,NaN,NaN,1
1,1981-01,"Halifax, Nova Scotia",2021S0503205,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617909756,5.10.1,32.4,NaN,NaN,NaN,1
2,1981-01,"Montréal, Quebec",2021S0503462,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617911676,10.10.1,24.6,NaN,NaN,NaN,1
3,1981-01,"Ottawa–Gatineau, Ontario part, Ontario/Quebec",2021S050535505,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617912444,12.10.1,17.9,NaN,NaN,NaN,1
4,1981-01,"Toronto, Ontario",2021S0503535,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617912828,13.10.1,16.2,NaN,NaN,NaN,1


,Event ID,Event Date,Event Name,Region,Peril,Industry Loss (CAD),Estimate Type,Primary Source,Notes,Unnamed: 9,Year,Event,Latest insured loss,Confidence
0,AB20200613,6/13/2020,Calgary Hailstorm,Calgary,Hail / Convective Storm,"1,300,000,000",Final available,IBC (CatIQ),One of Canada's costliest hail events.,NaN,2020,Calgary hailstorm,~$1.2–1.3B,?????
1,AB20210620*,2021 (date to verify),Calgary Hailstorm,Calgary,Hail / Convective Storm,"700,000,000+",Final available,IBC (CatIQ),Mentioned by IBC as a major recent Calgary hai...,NaN,2021,Calgary hailstorm & flooding (July 2),>$700M,?????
2,AB20230715,7/15/2023,South Calgary Hailstorm,Calgary,Hail / Convective Storm,"110,000,000",Final available,IBC,"Damaged homes, vehicles and commercial property.",NaN,2023,South Calgary hailstorm,~$110M,????
3,AB20240805,8/5/2024,Calgary Hailstorm,Calgary,Hail / Convective Storm,"3,293,000,000",Latest available,PERILS / CatIQ,Record revised estimate after claims development.,NaN,2024,Calgary hailstorm,$3.293B,?????
4,AB20250713,7/13/2025,Calgary Hailstorm,Calgary,Hail / Convective Storm,"164,000,000",Latest available,IBC (CatIQ),"Initial estimate $92M, later revised to $164M.",NaN,2025 Jul,Calgary hailstorm,$164M,?????


In [10]:
pd.read_csv(files["construction"], sep="\t", encoding="cp1252")

,Unnamed: 0,GEO,DGUID,Type of building,Division,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,1981-01,Fifteen census metropolitan area composite,NaN,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617908220,1.10.1,23.5,NaN,NaN,NaN,1
1,1981-01,"Halifax, Nova Scotia",2021S0503205,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617909756,5.10.1,32.4,NaN,NaN,NaN,1
2,1981-01,"Montréal, Quebec",2021S0503462,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617911676,10.10.1,24.6,NaN,NaN,NaN,1
3,1981-01,"Ottawa–Gatineau, Ontario part, Ontario/Quebec",2021S050535505,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617912444,12.10.1,17.9,NaN,NaN,NaN,1
4,1981-01,"Toronto, Ontario",2021S0503535,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617912828,13.10.1,16.2,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301449,2026-01,"Victoria, British Columbia",2021S0503935,Apartment buildings,Communications,"Index, 2023=100",451,units,0,v1617917263,25.2.20,109.1,NaN,NaN,NaN,1
301450,2026-01,"Victoria, British Columbia",2021S0503935,Apartment buildings,Electrical safety and security,"Index, 2023=100",451,units,0,v1617917264,25.2.21,108.6,NaN,NaN,NaN,1
301451,2026-01,"Victoria, British Columbia",2021S0503935,Apartment buildings,Earthwork,"Index, 2023=100",451,units,0,v1617917265,25.2.22,132.2,NaN,NaN,NaN,1
301452,2026-01,"Victoria, British Columbia",2021S0503935,Apartment buildings,Exterior improvements,"Index, 2023=100",451,units,0,v1617917266,25.2.23,113.6,NaN,NaN,NaN,1


In [11]:
construction_file = raw_data_path / "18100289.csv"

construction_raw = pd.read_csv(
    construction_file,
    sep="\t",
    encoding="latin1",
    low_memory=False
)

print("Construction file loaded.")
print("Construction shape:", construction_raw.shape)
display(construction_raw.head())

Construction file loaded.
Construction shape: (301454, 16)


,Unnamed: 0,GEO,DGUID,Type of building,Division,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,1981-01,Fifteen census metropolitan area composite,NaN,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617908220,1.10.1,23.5,NaN,NaN,NaN,1
1,1981-01,"Halifax, Nova Scotia",2021S0503205,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617909756,5.10.1,32.4,NaN,NaN,NaN,1
2,1981-01,"Montréal, Quebec",2021S0503462,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617911676,10.10.1,24.6,NaN,NaN,NaN,1
3,1981-01,"OttawaGatineau, Ontario part, Ontario/Quebec",2021S050535505,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617912444,12.10.1,17.9,NaN,NaN,NaN,1
4,1981-01,"Toronto, Ontario",2021S0503535,Warehouse [62212],Division composite,"Index, 2023=100",451,units,0,v1617912828,13.10.1,16.2,NaN,NaN,NaN,1


In [12]:
print("Construction shape:", construction_raw.shape)

print("\nGeographies containing Calgary, Edmonton, Alberta, or composite:")
display(
    construction_raw[
        construction_raw["GEO"]
        .astype(str)
        .str.contains("Calgary|Edmonton|Alberta|composite", case=False, na=False)
    ]["GEO"]
    .drop_duplicates()
    .sort_values()
    .to_frame()
)

print("\nBuilding types available:")
display(
    construction_raw["Type of building"]
    .drop_duplicates()
    .sort_values()
    .to_frame()
)

print("\nDivision values available:")
display(
    construction_raw["Division"]
    .drop_duplicates()
    .sort_values()
    .to_frame()
)

Construction shape: (301454, 16)

Geographies containing Calgary, Edmonton, Alberta, or composite:


,GEO
1472,Alberta
5,"Calgary, Alberta"
6,"Edmonton, Alberta"
0,Fifteen census metropolitan area composite



Building types available:


,Type of building
281558,Apartment buildings
263320,Bus depot with maintenance and repair facilities
243101,Commercial buildings [62212]
224540,Factory
204537,High-rise apartment building (five or more sto...
185147,Industrial buildings [62211]
164928,Institutional buildings [62213]
145861,Low-rise apartment building (fewer than five s...
125642,Non-residential buildings [622]
105423,Office building [62212]



Division values available:


,Division
1167,Communications
1154,Concrete
66472,Conveying equipment
85529,Demolition
0,Division composite
1169,Earthwork
1166,Electrical
1168,Electrical safety and security
1161,Equipment
1170,Exterior improvements


In [13]:
import pandas as pd
from pathlib import Path

project_root = Path("C:/Projects/AnalyticsPortfolio")
raw_data_path = project_root / "data" / "raw"

construction_file = raw_data_path / "18100289.csv"

construction_raw = pd.read_csv(
    construction_file,
    sep="\t",
    encoding="cp1252"
)

print("Construction file loaded.")
print("Construction shape:", construction_raw.shape)

Construction file loaded.
Construction shape: (301454, 16)


In [14]:
print("Construction shape:", construction_raw.shape)

print("\nGeographies containing Calgary, Edmonton, Alberta, or composite:")
display(
    construction_raw[
        construction_raw["GEO"]
        .astype(str)
        .str.contains("Calgary|Edmonton|Alberta|composite", case=False, na=False)
    ]["GEO"]
    .drop_duplicates()
    .sort_values()
    .to_frame()
)

print("\nBuilding types available:")
display(
    construction_raw["Type of building"]
    .drop_duplicates()
    .sort_values()
    .to_frame()
)

print("\nDivision values available:")
display(
    construction_raw["Division"]
    .drop_duplicates()
    .sort_values()
    .to_frame()
)

Construction shape: (301454, 16)

Geographies containing Calgary, Edmonton, Alberta, or composite:


,GEO
1472,Alberta
5,"Calgary, Alberta"
6,"Edmonton, Alberta"
0,Fifteen census metropolitan area composite



Building types available:


,Type of building
281558,Apartment buildings
263320,Bus depot with maintenance and repair facilities
243101,Commercial buildings [62212]
224540,Factory
204537,High-rise apartment building (five or more sto...
185147,Industrial buildings [62211]
164928,Institutional buildings [62213]
145861,Low-rise apartment building (fewer than five s...
125642,Non-residential buildings [622]
105423,Office building [62212]



Division values available:


,Division
1167,Communications
1154,Concrete
66472,Conveying equipment
85529,Demolition
0,Division composite
1169,Earthwork
1166,Electrical
1168,Electrical safety and security
1161,Equipment
1170,Exterior improvements


In [15]:
import pandas as pd
from pathlib import Path

project_root = Path("C:/Projects/AnalyticsPortfolio")
raw_data_path = project_root / "data" / "raw"
cpi_file = raw_data_path / "1810000401_databaseLoadingData.csv"

cpi_raw = pd.read_csv(
    cpi_file,
    encoding="utf-8-sig",
    low_memory=False
)

print("CPI file loaded.")
print("CPI shape:", cpi_raw.shape)
display(cpi_raw.head())


print("\nRelevant geographies:")
display(
    cpi_raw[
        cpi_raw["GEO"]
        .astype(str)
        .str.contains("Alberta|Canada", case=False, na=False)
    ]["GEO"]
    .drop_duplicates()
    .sort_values()
    .to_frame()
)

print("\nInsurance-related product categories:")
display(
    cpi_raw[
        cpi_raw["Products and product groups"]
        .astype(str)
        .str.contains("insurance", case=False, na=False)
    ]["Products and product groups"]
    .drop_duplicates()
    .sort_values()
    .to_frame()
)

print("\nAll-items categories:")
display(
    cpi_raw[
        cpi_raw["Products and product groups"]
        .astype(str)
        .str.contains("All-items", case=False, na=False)
    ]["Products and product groups"]
    .drop_duplicates()
    .sort_values()
    .to_frame()
)

CPI file loaded.
CPI shape: (2400, 15)


,REF_DATE,GEO,DGUID,Products and product groups,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2016-01,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,126.8,NaN,NaN,NaN,1
1,2016-02,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,127.1,NaN,NaN,NaN,1
2,2016-03,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,127.9,NaN,NaN,NaN,1
3,2016-04,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,128.3,NaN,NaN,NaN,1
4,2016-05,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,128.8,NaN,NaN,NaN,1



Relevant geographies:


,GEO
1680,Alberta
2040,"Calgary, Alberta"
0,Canada
1920,"Edmonton, Alberta"



Insurance-related product categories:


,Products and product groups
120,Homeowners' home and mortgage insurance



All-items categories:


,Products and product groups
0,All-items


In [16]:
import pandas as pd
from pathlib import Path

project_root = Path("C:/Projects/AnalyticsPortfolio")
raw_data_path = project_root / "data" / "raw"

cpi_file = raw_data_path / "1810000401_databaseLoadingData.csv"

cpi_raw = pd.read_csv(
    cpi_file,
    encoding="utf-8-sig",
    low_memory=False
)

print("CPI file loaded.")
print("CPI shape:", cpi_raw.shape)
display(cpi_raw.head())

print("\nRelevant geographies:")
display(
    cpi_raw[
        cpi_raw["GEO"]
        .astype(str)
        .str.contains("Alberta|Canada", case=False, na=False)
    ]["GEO"]
    .drop_duplicates()
    .sort_values()
    .to_frame()
)

print("\nInsurance-related product categories:")
display(
    cpi_raw[
        cpi_raw["Products and product groups"]
        .astype(str)
        .str.contains("insurance", case=False, na=False)
    ]["Products and product groups"]
    .drop_duplicates()
    .sort_values()
    .to_frame()
)

print("\nAll-items categories:")
display(
    cpi_raw[
        cpi_raw["Products and product groups"]
        .astype(str)
        .str.contains("All-items", case=False, na=False)
    ]["Products and product groups"]
    .drop_duplicates()
    .sort_values()
    .to_frame()
)

CPI file loaded.
CPI shape: (2400, 15)


,REF_DATE,GEO,DGUID,Products and product groups,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2016-01,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,126.8,NaN,NaN,NaN,1
1,2016-02,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,127.1,NaN,NaN,NaN,1
2,2016-03,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,127.9,NaN,NaN,NaN,1
3,2016-04,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,128.3,NaN,NaN,NaN,1
4,2016-05,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,128.8,NaN,NaN,NaN,1



Relevant geographies:


,GEO
1680,Alberta
2040,"Calgary, Alberta"
0,Canada
1920,"Edmonton, Alberta"



Insurance-related product categories:


,Products and product groups
120,Homeowners' home and mortgage insurance



All-items categories:


,Products and product groups
0,All-items


In [17]:
import pandas as pd
from pathlib import Path

project_root = Path("C:/Projects/AnalyticsPortfolio")
raw_data_path = project_root / "data" / "raw"

cpi_file = raw_data_path / "1810000401_databaseLoadingData.csv"

# Load only the columns we need
cpi_raw = pd.read_csv(
    cpi_file,
    encoding="utf-8-sig",
    usecols=["REF_DATE", "GEO", "Products and product groups", "VALUE"],
    low_memory=False
)

# Define filters
cpi_geographies = ["Alberta", "Canada"]
cpi_categories = [
    "Homeowners' home and mortgage insurance",
    "All-items"
]

# Filter rows
cpi_clean = cpi_raw[
    (cpi_raw["GEO"].isin(cpi_geographies)) &
    (cpi_raw["Products and product groups"].isin(cpi_categories))
].copy()

# Keep only the combinations we need:
# Alberta insurance, Canada insurance, Alberta all-items
cpi_clean = cpi_clean[
    (
        (cpi_clean["GEO"] == "Alberta") &
        (cpi_clean["Products and product groups"].isin([
            "Homeowners' home and mortgage insurance",
            "All-items"
        ]))
    )
    |
    (
        (cpi_clean["GEO"] == "Canada") &
        (cpi_clean["Products and product groups"] == "Homeowners' home and mortgage insurance")
    )
].copy()

# Rename columns for warehouse
cpi_clean = cpi_clean.rename(columns={
    "REF_DATE": "year_month",
    "GEO": "geography",
    "Products and product groups": "cpi_category",
    "VALUE": "cpi_value"
})

# Create date_id
cpi_clean["date_id"] = cpi_clean["year_month"].str.replace("-", "").astype(int)

# Add source metadata
cpi_clean["source_table"] = "18-10-0004"
cpi_clean["source_name"] = "Statistics Canada Consumer Price Index"

# Reorder columns to match warehouse table
cpi_clean = cpi_clean[
    [
        "date_id",
        "geography",
        "cpi_category",
        "cpi_value",
        "source_table",
        "source_name"
    ]
]

print("CPI clean shape:", cpi_clean.shape)
display(cpi_clean.head())
display(cpi_clean.tail())

print("\nRows by geography and category:")
display(
    cpi_clean
    .groupby(["geography", "cpi_category"])
    .size()
    .reset_index(name="row_count")
)

print("\nCPI date range:")
display(
    cpi_clean
    .groupby(["geography", "cpi_category"])
    .agg(
        min_date_id=("date_id", "min"),
        max_date_id=("date_id", "max"),
        row_count=("date_id", "count")
    )
    .reset_index()
)

CPI clean shape: (360, 6)


,date_id,geography,cpi_category,cpi_value,source_table,source_name
120,201601,Canada,Homeowners' home and mortgage insurance,217.4,18-10-0004,Statistics Canada Consumer Price Index
121,201602,Canada,Homeowners' home and mortgage insurance,216.4,18-10-0004,Statistics Canada Consumer Price Index
122,201603,Canada,Homeowners' home and mortgage insurance,215.8,18-10-0004,Statistics Canada Consumer Price Index
123,201604,Canada,Homeowners' home and mortgage insurance,216.5,18-10-0004,Statistics Canada Consumer Price Index
124,201605,Canada,Homeowners' home and mortgage insurance,216.7,18-10-0004,Statistics Canada Consumer Price Index


,date_id,geography,cpi_category,cpi_value,source_table,source_name
1915,202508,Alberta,Homeowners' home and mortgage insurance,634.3,18-10-0004,Statistics Canada Consumer Price Index
1916,202509,Alberta,Homeowners' home and mortgage insurance,642.1,18-10-0004,Statistics Canada Consumer Price Index
1917,202510,Alberta,Homeowners' home and mortgage insurance,646.6,18-10-0004,Statistics Canada Consumer Price Index
1918,202511,Alberta,Homeowners' home and mortgage insurance,647.1,18-10-0004,Statistics Canada Consumer Price Index
1919,202512,Alberta,Homeowners' home and mortgage insurance,647.4,18-10-0004,Statistics Canada Consumer Price Index



Rows by geography and category:


,geography,cpi_category,row_count
0,Alberta,All-items,120
1,Alberta,Homeowners' home and mortgage insurance,120
2,Canada,Homeowners' home and mortgage insurance,120



CPI date range:


,geography,cpi_category,min_date_id,max_date_id,row_count
0,Alberta,All-items,201601,202512,120
1,Alberta,Homeowners' home and mortgage insurance,201601,202512,120
2,Canada,Homeowners' home and mortgage insurance,201601,202512,120


In [18]:
import pandas as pd
from pathlib import Path

project_root = Path("C:/Projects/AnalyticsPortfolio")
raw_data_path = project_root / "data" / "raw"

cpi_file = raw_data_path / "1810000401_databaseLoadingData.csv"

# Load only the columns we need
cpi_raw = pd.read_csv(
    cpi_file,
    encoding="utf-8-sig",
    usecols=["REF_DATE", "GEO", "Products and product groups", "VALUE"],
    low_memory=False
)

# Define filters
cpi_geographies = ["Alberta", "Canada"]
cpi_categories = [
    "Homeowners' home and mortgage insurance",
    "All-items"
]

# Filter rows
cpi_clean = cpi_raw[
    (cpi_raw["GEO"].isin(cpi_geographies)) &
    (cpi_raw["Products and product groups"].isin(cpi_categories))
].copy()

# Keep only the combinations we need:
# Alberta insurance, Canada insurance, Alberta all-items
cpi_clean = cpi_clean[
    (
        (cpi_clean["GEO"] == "Alberta") &
        (cpi_clean["Products and product groups"].isin([
            "Homeowners' home and mortgage insurance",
            "All-items"
        ]))
    )
    |
    (
        (cpi_clean["GEO"] == "Canada") &
        (cpi_clean["Products and product groups"] == "Homeowners' home and mortgage insurance")
    )
].copy()

# Rename columns for warehouse
cpi_clean = cpi_clean.rename(columns={
    "REF_DATE": "year_month",
    "GEO": "geography",
    "Products and product groups": "cpi_category",
    "VALUE": "cpi_value"
})

# Create date_id
cpi_clean["date_id"] = cpi_clean["year_month"].str.replace("-", "").astype(int)

# Add source metadata
cpi_clean["source_table"] = "18-10-0004"
cpi_clean["source_name"] = "Statistics Canada Consumer Price Index"

# Reorder columns to match warehouse table
cpi_clean = cpi_clean[
    [
        "date_id",
        "geography",
        "cpi_category",
        "cpi_value",
        "source_table",
        "source_name"
    ]
]

print("CPI clean shape:", cpi_clean.shape)
display(cpi_clean.head())
display(cpi_clean.tail())

print("\nRows by geography and category:")
display(
    cpi_clean
    .groupby(["geography", "cpi_category"])
    .size()
    .reset_index(name="row_count")
)

print("\nCPI date range:")
display(
    cpi_clean
    .groupby(["geography", "cpi_category"])
    .agg(
        min_date_id=("date_id", "min"),
        max_date_id=("date_id", "max"),
        row_count=("date_id", "count")
    )
    .reset_index()
)

CPI clean shape: (360, 6)


,date_id,geography,cpi_category,cpi_value,source_table,source_name
120,201601,Canada,Homeowners' home and mortgage insurance,217.4,18-10-0004,Statistics Canada Consumer Price Index
121,201602,Canada,Homeowners' home and mortgage insurance,216.4,18-10-0004,Statistics Canada Consumer Price Index
122,201603,Canada,Homeowners' home and mortgage insurance,215.8,18-10-0004,Statistics Canada Consumer Price Index
123,201604,Canada,Homeowners' home and mortgage insurance,216.5,18-10-0004,Statistics Canada Consumer Price Index
124,201605,Canada,Homeowners' home and mortgage insurance,216.7,18-10-0004,Statistics Canada Consumer Price Index


,date_id,geography,cpi_category,cpi_value,source_table,source_name
1915,202508,Alberta,Homeowners' home and mortgage insurance,634.3,18-10-0004,Statistics Canada Consumer Price Index
1916,202509,Alberta,Homeowners' home and mortgage insurance,642.1,18-10-0004,Statistics Canada Consumer Price Index
1917,202510,Alberta,Homeowners' home and mortgage insurance,646.6,18-10-0004,Statistics Canada Consumer Price Index
1918,202511,Alberta,Homeowners' home and mortgage insurance,647.1,18-10-0004,Statistics Canada Consumer Price Index
1919,202512,Alberta,Homeowners' home and mortgage insurance,647.4,18-10-0004,Statistics Canada Consumer Price Index



Rows by geography and category:


,geography,cpi_category,row_count
0,Alberta,All-items,120
1,Alberta,Homeowners' home and mortgage insurance,120
2,Canada,Homeowners' home and mortgage insurance,120



CPI date range:


,geography,cpi_category,min_date_id,max_date_id,row_count
0,Alberta,All-items,201601,202512,120
1,Alberta,Homeowners' home and mortgage insurance,201601,202512,120
2,Canada,Homeowners' home and mortgage insurance,201601,202512,120


In [19]:
import pandas as pd
from pathlib import Path

project_root = Path("C:/Projects/AnalyticsPortfolio")
raw_data_path = project_root / "data" / "raw"

cpi_file = raw_data_path / "1810000401_databaseLoadingData.csv"

# Load only the columns we need
cpi_raw = pd.read_csv(
    cpi_file,
    encoding="utf-8-sig",
    usecols=["REF_DATE", "GEO", "Products and product groups", "VALUE"],
    low_memory=False
)

# Define filters
cpi_geographies = ["Alberta", "Canada"]
cpi_categories = [
    "Homeowners' home and mortgage insurance",
    "All-items"
]

# Filter rows
cpi_clean = cpi_raw[
    (cpi_raw["GEO"].isin(cpi_geographies)) &
    (cpi_raw["Products and product groups"].isin(cpi_categories))
].copy()

# Keep only the combinations we need:
# Alberta insurance, Canada insurance, Alberta all-items
cpi_clean = cpi_clean[
    (
        (cpi_clean["GEO"] == "Alberta") &
        (cpi_clean["Products and product groups"].isin([
            "Homeowners' home and mortgage insurance",
            "All-items"
        ]))
    )
    |
    (
        (cpi_clean["GEO"] == "Canada") &
        (cpi_clean["Products and product groups"] == "Homeowners' home and mortgage insurance")
    )
].copy()

# Rename columns for warehouse
cpi_clean = cpi_clean.rename(columns={
    "REF_DATE": "year_month",
    "GEO": "geography",
    "Products and product groups": "cpi_category",
    "VALUE": "cpi_value"
})

# Create date_id
cpi_clean["date_id"] = cpi_clean["year_month"].str.replace("-", "").astype(int)

# Add source metadata
cpi_clean["source_table"] = "18-10-0004"
cpi_clean["source_name"] = "Statistics Canada Consumer Price Index"

# Reorder columns to match warehouse table
cpi_clean = cpi_clean[
    [
        "date_id",
        "geography",
        "cpi_category",
        "cpi_value",
        "source_table",
        "source_name"
    ]
]

print("CPI clean shape:", cpi_clean.shape)
display(cpi_clean.head())
display(cpi_clean.tail())

print("\nRows by geography and category:")
display(
    cpi_clean
    .groupby(["geography", "cpi_category"])
    .size()
    .reset_index(name="row_count")
)

print("\nCPI date range:")
display(
    cpi_clean
    .groupby(["geography", "cpi_category"])
    .agg(
        min_date_id=("date_id", "min"),
        max_date_id=("date_id", "max"),
        row_count=("date_id", "count")
    )
    .reset_index()
)

CPI clean shape: (360, 6)


,date_id,geography,cpi_category,cpi_value,source_table,source_name
120,201601,Canada,Homeowners' home and mortgage insurance,217.4,18-10-0004,Statistics Canada Consumer Price Index
121,201602,Canada,Homeowners' home and mortgage insurance,216.4,18-10-0004,Statistics Canada Consumer Price Index
122,201603,Canada,Homeowners' home and mortgage insurance,215.8,18-10-0004,Statistics Canada Consumer Price Index
123,201604,Canada,Homeowners' home and mortgage insurance,216.5,18-10-0004,Statistics Canada Consumer Price Index
124,201605,Canada,Homeowners' home and mortgage insurance,216.7,18-10-0004,Statistics Canada Consumer Price Index


,date_id,geography,cpi_category,cpi_value,source_table,source_name
1915,202508,Alberta,Homeowners' home and mortgage insurance,634.3,18-10-0004,Statistics Canada Consumer Price Index
1916,202509,Alberta,Homeowners' home and mortgage insurance,642.1,18-10-0004,Statistics Canada Consumer Price Index
1917,202510,Alberta,Homeowners' home and mortgage insurance,646.6,18-10-0004,Statistics Canada Consumer Price Index
1918,202511,Alberta,Homeowners' home and mortgage insurance,647.1,18-10-0004,Statistics Canada Consumer Price Index
1919,202512,Alberta,Homeowners' home and mortgage insurance,647.4,18-10-0004,Statistics Canada Consumer Price Index



Rows by geography and category:


,geography,cpi_category,row_count
0,Alberta,All-items,120
1,Alberta,Homeowners' home and mortgage insurance,120
2,Canada,Homeowners' home and mortgage insurance,120



CPI date range:


,geography,cpi_category,min_date_id,max_date_id,row_count
0,Alberta,All-items,201601,202512,120
1,Alberta,Homeowners' home and mortgage insurance,201601,202512,120
2,Canada,Homeowners' home and mortgage insurance,201601,202512,120


In [20]:
import pandas as pd

cpi_test = pd.read_csv(
    cpi_file,
    encoding="cp1252",
    nrows=5
)

print("Columns:")
print(list(cpi_test.columns))

display(cpi_test)

Columns:
['ï»¿"REF_DATE"', 'GEO', 'DGUID', 'Products and product groups', 'UOM', 'UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE', 'STATUS', 'SYMBOL', 'TERMINATED', 'DECIMALS']


,"ï»¿""REF_DATE""",GEO,DGUID,Products and product groups,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2016-01,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,126.8,NaN,NaN,NaN,1
1,2016-02,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,127.1,NaN,NaN,NaN,1
2,2016-03,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,127.9,NaN,NaN,NaN,1
3,2016-04,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,128.3,NaN,NaN,NaN,1
4,2016-05,Canada,2016A000011124,All-items,2002=100,17,units,0,v41690973,2.2,128.8,NaN,NaN,NaN,1


In [21]:
cpi_test_tab = pd.read_csv(
    cpi_file,
    sep="\t",
    encoding="cp1252",
    nrows=5
)

print("Tab columns:")
print(list(cpi_test_tab.columns))

display(cpi_test_tab)

Tab columns:
['ï»¿"REF_DATE","GEO","DGUID","Products and product groups","UOM","UOM_ID","SCALAR_FACTOR","SCALAR_ID","VECTOR","COORDINATE","VALUE","STATUS","SYMBOL","TERMINATED","DECIMALS"']


,"ï»¿""REF_DATE"",""GEO"",""DGUID"",""Products and product groups"",""UOM"",""UOM_ID"",""SCALAR_FACTOR"",""SCALAR_ID"",""VECTOR"",""COORDINATE"",""VALUE"",""STATUS"",""SYMBOL"",""TERMINATED"",""DECIMALS"""
0,"2016-01,""Canada"",""2016A000011124"",""All-items"",..."
1,"2016-02,""Canada"",""2016A000011124"",""All-items"",..."
2,"2016-03,""Canada"",""2016A000011124"",""All-items"",..."
3,"2016-04,""Canada"",""2016A000011124"",""All-items"",..."
4,"2016-05,""Canada"",""2016A000011124"",""All-items"",..."


In [22]:
import pandas as pd
from pathlib import Path

project_root = Path("C:/Projects/AnalyticsPortfolio")
raw_data_path = project_root / "data" / "raw"

cpi_file = raw_data_path / "1810000401_databaseLoadingData.csv"

cpi_raw = pd.read_csv(
    cpi_file,
    encoding="utf-8-sig",
    usecols=["REF_DATE", "GEO", "Products and product groups", "VALUE"],
    low_memory=False
)

print("CPI raw shape:", cpi_raw.shape)
display(cpi_raw.head())

CPI raw shape: (2400, 4)


,REF_DATE,GEO,Products and product groups,VALUE
0,2016-01,Canada,All-items,126.8
1,2016-02,Canada,All-items,127.1
2,2016-03,Canada,All-items,127.9
3,2016-04,Canada,All-items,128.3
4,2016-05,Canada,All-items,128.8


In [23]:
cpi_clean = cpi_raw[
    (
        (cpi_raw["GEO"] == "Alberta") &
        (cpi_raw["Products and product groups"].isin([
            "All-items",
            "Homeowners' home and mortgage insurance"
        ]))
    )
    |
    (
        (cpi_raw["GEO"] == "Canada") &
        (cpi_raw["Products and product groups"] == "Homeowners' home and mortgage insurance")
    )
].copy()

cpi_clean = cpi_clean.rename(columns={
    "REF_DATE": "year_month",
    "GEO": "geography",
    "Products and product groups": "cpi_category",
    "VALUE": "cpi_value"
})

cpi_clean["date_id"] = cpi_clean["year_month"].str.replace("-", "").astype(int)

cpi_clean["source_table"] = "18-10-0004"
cpi_clean["source_name"] = "Statistics Canada Consumer Price Index"

cpi_clean = cpi_clean[
    [
        "date_id",
        "geography",
        "cpi_category",
        "cpi_value",
        "source_table",
        "source_name"
    ]
]

print("CPI clean shape:", cpi_clean.shape)
display(cpi_clean.head())
display(cpi_clean.tail())

CPI clean shape: (360, 6)


,date_id,geography,cpi_category,cpi_value,source_table,source_name
120,201601,Canada,Homeowners' home and mortgage insurance,217.4,18-10-0004,Statistics Canada Consumer Price Index
121,201602,Canada,Homeowners' home and mortgage insurance,216.4,18-10-0004,Statistics Canada Consumer Price Index
122,201603,Canada,Homeowners' home and mortgage insurance,215.8,18-10-0004,Statistics Canada Consumer Price Index
123,201604,Canada,Homeowners' home and mortgage insurance,216.5,18-10-0004,Statistics Canada Consumer Price Index
124,201605,Canada,Homeowners' home and mortgage insurance,216.7,18-10-0004,Statistics Canada Consumer Price Index


,date_id,geography,cpi_category,cpi_value,source_table,source_name
1915,202508,Alberta,Homeowners' home and mortgage insurance,634.3,18-10-0004,Statistics Canada Consumer Price Index
1916,202509,Alberta,Homeowners' home and mortgage insurance,642.1,18-10-0004,Statistics Canada Consumer Price Index
1917,202510,Alberta,Homeowners' home and mortgage insurance,646.6,18-10-0004,Statistics Canada Consumer Price Index
1918,202511,Alberta,Homeowners' home and mortgage insurance,647.1,18-10-0004,Statistics Canada Consumer Price Index
1919,202512,Alberta,Homeowners' home and mortgage insurance,647.4,18-10-0004,Statistics Canada Consumer Price Index


In [24]:
print("Rows by geography and category:")
display(
    cpi_clean
    .groupby(["geography", "cpi_category"])
    .size()
    .reset_index(name="row_count")
)

print("CPI date range:")
display(
    cpi_clean
    .groupby(["geography", "cpi_category"])
    .agg(
        min_date_id=("date_id", "min"),
        max_date_id=("date_id", "max"),
        row_count=("date_id", "count")
    )
    .reset_index()
)

Rows by geography and category:


,geography,cpi_category,row_count
0,Alberta,All-items,120
1,Alberta,Homeowners' home and mortgage insurance,120
2,Canada,Homeowners' home and mortgage insurance,120


CPI date range:


,geography,cpi_category,min_date_id,max_date_id,row_count
0,Alberta,All-items,201601,202512,120
1,Alberta,Homeowners' home and mortgage insurance,201601,202512,120
2,Canada,Homeowners' home and mortgage insurance,201601,202512,120


# Create dim_date from all date_ids used in cleaned dataframes

In [25]:
# Load raw construction data
construction_file = raw_data_path / "18100289.csv"

construction_raw = pd.read_csv(
    construction_file,
    sep="\t",
    encoding="latin1",
    low_memory=False
)

# Clean construction benchmark data
construction_clean = construction_raw[
    (construction_raw["GEO"] == "Alberta") &
    (construction_raw["Type of building"] == "Residential buildings [621]") &
    (construction_raw["Division"] == "Division composite")
].copy()

construction_clean = construction_clean.rename(columns={
    "Unnamed: 0": "year_month",
    "GEO": "geography",
    "Type of building": "construction_category",
    "VALUE": "construction_index_value"
})

construction_clean["date_id"] = construction_clean["year_month"].str.replace("-", "").astype(int)

construction_clean["frequency"] = "Quarterly"
construction_clean["geographic_proxy_flag"] = "Province-wide"
construction_clean["source_table"] = "18-10-0289"
construction_clean["source_name"] = "Statistics Canada Building Construction Price Index"

construction_clean = construction_clean[
    [
        "date_id",
        "geography",
        "construction_category",
        "construction_index_value",
        "frequency",
        "geographic_proxy_flag",
        "source_table",
        "source_name"
    ]
].sort_values("date_id")

print("construction_clean created:", construction_clean.shape)
display(construction_clean.head())
display(construction_clean.tail())

construction_clean created: (37, 8)


,date_id,geography,construction_category,construction_index_value,frequency,geographic_proxy_flag,source_table,source_name
85911,201701,Alberta,Residential buildings [621],56.9,Quarterly,Province-wide,18-10-0289,Statistics Canada Building Construction Price ...
86415,201704,Alberta,Residential buildings [621],57.4,Quarterly,Province-wide,18-10-0289,Statistics Canada Building Construction Price ...
86919,201707,Alberta,Residential buildings [621],58.5,Quarterly,Province-wide,18-10-0289,Statistics Canada Building Construction Price ...
87423,201710,Alberta,Residential buildings [621],59.4,Quarterly,Province-wide,18-10-0289,Statistics Canada Building Construction Price ...
87927,201801,Alberta,Residential buildings [621],59.9,Quarterly,Province-wide,18-10-0289,Statistics Canada Building Construction Price ...


,date_id,geography,construction_category,construction_index_value,frequency,geographic_proxy_flag,source_table,source_name
102879,202501,Alberta,Residential buildings [621],109.7,Quarterly,Province-wide,18-10-0289,Statistics Canada Building Construction Price ...
103479,202504,Alberta,Residential buildings [621],110.5,Quarterly,Province-wide,18-10-0289,Statistics Canada Building Construction Price ...
104079,202507,Alberta,Residential buildings [621],111.0,Quarterly,Province-wide,18-10-0289,Statistics Canada Building Construction Price ...
104679,202510,Alberta,Residential buildings [621],110.5,Quarterly,Province-wide,18-10-0289,Statistics Canada Building Construction Price ...
105279,202601,Alberta,Residential buildings [621],110.7,Quarterly,Province-wide,18-10-0289,Statistics Canada Building Construction Price ...


In [26]:
# Load raw catastrophe event data
events_file = raw_data_path / "alberta_catastrophe_events.csv"

events_raw = pd.read_csv(
    events_file,
    encoding="latin1"
)

print("Events raw shape:", events_raw.shape)
display(events_raw.head())

Events raw shape: (6, 14)


,Event ID,Event Date,Event Name,Region,Peril,Industry Loss (CAD),Estimate Type,Primary Source,Notes,Unnamed: 9,Year,Event,Latest insured loss,Confidence
0,AB20200613,6/13/2020,Calgary Hailstorm,Calgary,Hail / Convective Storm,"1,300,000,000",Final available,IBC (CatIQ),One of Canada's costliest hail events.,NaN,2020,Calgary hailstorm,~$1.21.3B,?????
1,AB20210620*,2021 (date to verify),Calgary Hailstorm,Calgary,Hail / Convective Storm,"700,000,000+",Final available,IBC (CatIQ),Mentioned by IBC as a major recent Calgary hai...,NaN,2021,Calgary hailstorm & flooding (July 2),>$700M,?????
2,AB20230715,7/15/2023,South Calgary Hailstorm,Calgary,Hail / Convective Storm,"110,000,000",Final available,IBC,"Damaged homes, vehicles and commercial property.",NaN,2023,South Calgary hailstorm,~$110M,????
3,AB20240805,8/5/2024,Calgary Hailstorm,Calgary,Hail / Convective Storm,"3,293,000,000",Latest available,PERILS / CatIQ,Record revised estimate after claims development.,NaN,2024,Calgary hailstorm,$3.293B,?????
4,AB20250713,7/13/2025,Calgary Hailstorm,Calgary,Hail / Convective Storm,"164,000,000",Latest available,IBC (CatIQ),"Initial estimate $92M, later revised to $164M.",NaN,2025 Jul,Calgary hailstorm,$164M,?????


In [27]:
# Clean curated catastrophe event data
events_clean = events_raw.copy()

# Use manually corrected event dates from the final curated file
events_clean["event_date_parsed"] = pd.to_datetime(
    events_clean["Event Date"],
    errors="coerce"
)

# If July 2, 2021 is stored as text but did not parse earlier, force-correct it
events_clean.loc[
    events_clean["Event ID"].astype(str).str.contains("2021", na=False),
    "event_date_parsed"
] = pd.to_datetime("2021-07-02")

# Keep only rows with valid parsed event dates
events_clean = events_clean[
    events_clean["event_date_parsed"].notna()
].copy()

# Create clean warehouse fields
events_clean["event_id"] = range(1, len(events_clean) + 1)
events_clean["event_date"] = events_clean["event_date_parsed"].dt.date
events_clean["date_id"] = events_clean["event_date_parsed"].dt.strftime("%Y%m").astype(int)

events_clean["event_name"] = events_clean["Event Name"]
events_clean["primary_region"] = events_clean["Region"]
events_clean["province"] = "Alberta"
events_clean["peril_type"] = events_clean["Peril"]
events_clean["initial_loss_cad"] = events_clean["Industry Loss (CAD)"]

# Convert loss strings like "700,000,000+" into numeric values
events_clean["revised_loss_cad"] = (
    events_clean["Industry Loss (CAD)"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
    .str.extract(r"(\d+\.?\d*)")[0]
    .astype(float)
)

events_clean["claim_count"] = None
events_clean["estimate_publication_date"] = None
events_clean["loss_source"] = events_clean["Primary Source"]
events_clean["source_url"] = None

events_clean["revision_flag"] = events_clean["Estimate Type"].str.contains(
    "latest|revised",
    case=False,
    na=False
).astype(int)

events_clean["notes"] = events_clean["Notes"]

events_clean = events_clean[
    [
        "event_id",
        "event_date",
        "date_id",
        "event_name",
        "primary_region",
        "province",
        "peril_type",
        "initial_loss_cad",
        "revised_loss_cad",
        "claim_count",
        "estimate_publication_date",
        "loss_source",
        "source_url",
        "revision_flag",
        "notes"
    ]
].sort_values("event_date")

print("events_clean created:", events_clean.shape)
display(events_clean)

events_clean created: (6, 15)


,event_id,event_date,date_id,event_name,primary_region,province,peril_type,initial_loss_cad,revised_loss_cad,claim_count,estimate_publication_date,loss_source,source_url,revision_flag,notes
0,1,2020-06-13,202006,Calgary Hailstorm,Calgary,Alberta,Hail / Convective Storm,"1,300,000,000",1.300000e+09,None,None,IBC (CatIQ),None,0,One of Canada's costliest hail events.
1,2,2021-07-02,202107,Calgary Hailstorm,Calgary,Alberta,Hail / Convective Storm,"700,000,000+",7.000000e+08,None,None,IBC (CatIQ),None,0,Mentioned by IBC as a major recent Calgary hai...
2,3,2023-07-15,202307,South Calgary Hailstorm,Calgary,Alberta,Hail / Convective Storm,"110,000,000",1.100000e+08,None,None,IBC,None,0,"Damaged homes, vehicles and commercial property."
3,4,2024-08-05,202408,Calgary Hailstorm,Calgary,Alberta,Hail / Convective Storm,"3,293,000,000",3.293000e+09,None,None,PERILS / CatIQ,None,1,Record revised estimate after claims development.
4,5,2025-07-13,202507,Calgary Hailstorm,Calgary,Alberta,Hail / Convective Storm,"164,000,000",1.640000e+08,None,None,IBC (CatIQ),None,1,"Initial estimate $92M, later revised to $164M."
5,6,2025-08-20,202508,Brooks Severe Storm,Brooks,Alberta,Severe Convective Storm,"235,000,000+",2.350000e+08,None,None,IBC (CatIQ),None,0,Included because it is a major Alberta insured...


In [28]:
all_date_ids = pd.concat([
    cpi_clean["date_id"],
    construction_clean["date_id"],
    events_clean["date_id"]
]).drop_duplicates().sort_values()

dim_date_clean = pd.DataFrame({"date_id": all_date_ids})

dim_date_clean["year_month"] = dim_date_clean["date_id"].astype(str).str[:4] + "-" + dim_date_clean["date_id"].astype(str).str[4:6]
dim_date_clean["calendar_date"] = dim_date_clean["year_month"] + "-01"
dim_date_clean["year"] = dim_date_clean["date_id"].astype(str).str[:4].astype(int)
dim_date_clean["month_number"] = dim_date_clean["date_id"].astype(str).str[4:6].astype(int)
dim_date_clean["quarter"] = ((dim_date_clean["month_number"] - 1) // 3) + 1

month_names = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}

dim_date_clean["month_name"] = dim_date_clean["month_number"].map(month_names)

dim_date_clean["five_year_cohort"] = dim_date_clean["year"].apply(
    lambda year: "2016-2020" if 2016 <= year <= 2020
    else "2021-2025" if 2021 <= year <= 2025
    else "Outside project window"
)

dim_date_clean = dim_date_clean[
    [
        "date_id",
        "calendar_date",
        "year_month",
        "year",
        "quarter",
        "month_number",
        "month_name",
        "five_year_cohort"
    ]
]

print("dim_date clean shape:", dim_date_clean.shape)
display(dim_date_clean.head())
display(dim_date_clean.tail())

print("Date dimension range:")
display(
    dim_date_clean
    .agg(
        min_date_id=("date_id", "min"),
        max_date_id=("date_id", "max"),
        row_count=("date_id", "count")
    )
)

dim_date clean shape: (121, 8)


,date_id,calendar_date,year_month,year,quarter,month_number,month_name,five_year_cohort
120,201601,2016-01-01,2016-01,2016,1,1,January,2016-2020
121,201602,2016-02-01,2016-02,2016,1,2,February,2016-2020
122,201603,2016-03-01,2016-03,2016,1,3,March,2016-2020
123,201604,2016-04-01,2016-04,2016,2,4,April,2016-2020
124,201605,2016-05-01,2016-05,2016,2,5,May,2016-2020


,date_id,calendar_date,year_month,year,quarter,month_number,month_name,five_year_cohort
236,202509,2025-09-01,2025-09,2025,3,9,September,2021-2025
237,202510,2025-10-01,2025-10,2025,4,10,October,2021-2025
238,202511,2025-11-01,2025-11,2025,4,11,November,2021-2025
239,202512,2025-12-01,2025-12,2025,4,12,December,2021-2025
105279,202601,2026-01-01,2026-01,2026,1,1,January,Outside project window


Date dimension range:


,date_id
min_date_id,201601
max_date_id,202601
row_count,121


In [29]:
print("CPI date range:")
display(
    cpi_clean
    .groupby(["geography", "cpi_category"])
    .agg(
        min_date_id=("date_id", "min"),
        max_date_id=("date_id", "max"),
        row_count=("date_id", "count")
    )
    .reset_index()
)

print("\nConstruction date range:")
display(
    construction_clean
    .groupby(["geography", "construction_category"])
    .agg(
        min_date_id=("date_id", "min"),
        max_date_id=("date_id", "max"),
        row_count=("date_id", "count")
    )
    .reset_index()
)

CPI date range:


,geography,cpi_category,min_date_id,max_date_id,row_count
0,Alberta,All-items,201601,202512,120
1,Alberta,Homeowners' home and mortgage insurance,201601,202512,120
2,Canada,Homeowners' home and mortgage insurance,201601,202512,120



Construction date range:


,geography,construction_category,min_date_id,max_date_id,row_count
0,Alberta,Residential buildings [621],201701,202601,37


In [30]:
# Check which key Phase 2 variables currently exist in memory
for var_name in ["cpi_clean", "construction_clean", "events_raw", "events_clean", "dim_date_clean"]:
    print(var_name, "exists:" , var_name in globals())

cpi_clean exists: True
construction_clean exists: True
events_raw exists: True
events_clean exists: True
dim_date_clean exists: True


In [31]:
import numpy as np

In [32]:
# Create date dimension from all cleaned fact-table date_ids
all_date_ids = pd.concat([
    cpi_clean["date_id"],
    construction_clean["date_id"],
    events_clean["date_id"]
]).drop_duplicates().sort_values()

dim_date_clean = pd.DataFrame({
    "date_id": all_date_ids
})

dim_date_clean["year_month"] = dim_date_clean["date_id"].astype(str).str[:4] + "-" + dim_date_clean["date_id"].astype(str).str[4:6]
dim_date_clean["calendar_date"] = pd.to_datetime(dim_date_clean["year_month"] + "-01")
dim_date_clean["year"] = dim_date_clean["calendar_date"].dt.year
dim_date_clean["quarter"] = dim_date_clean["calendar_date"].dt.quarter
dim_date_clean["month_number"] = dim_date_clean["calendar_date"].dt.month
dim_date_clean["month_name"] = dim_date_clean["calendar_date"].dt.month_name()

dim_date_clean["five_year_cohort"] = np.where(
    dim_date_clean["year"].between(2016, 2020),
    "2016-2020",
    np.where(
        dim_date_clean["year"].between(2021, 2025),
        "2021-2025",
        "Outside project window"
    )
)

dim_date_clean = dim_date_clean[
    [
        "date_id",
        "calendar_date",
        "year_month",
        "year",
        "quarter",
        "month_number",
        "month_name",
        "five_year_cohort"
    ]
].sort_values("date_id")

print("dim_date_clean created:", dim_date_clean.shape)
display(dim_date_clean.head())
display(dim_date_clean.tail())

print("Date dimension range:")
display(
    pd.DataFrame({
        "min_date_id": [dim_date_clean["date_id"].min()],
        "max_date_id": [dim_date_clean["date_id"].max()],
        "row_count": [dim_date_clean["date_id"].count()]
    })
)

dim_date_clean created: (121, 8)


,date_id,calendar_date,year_month,year,quarter,month_number,month_name,five_year_cohort
120,201601,2016-01-01,2016-01,2016,1,1,January,2016-2020
121,201602,2016-02-01,2016-02,2016,1,2,February,2016-2020
122,201603,2016-03-01,2016-03,2016,1,3,March,2016-2020
123,201604,2016-04-01,2016-04,2016,2,4,April,2016-2020
124,201605,2016-05-01,2016-05,2016,2,5,May,2016-2020


,date_id,calendar_date,year_month,year,quarter,month_number,month_name,five_year_cohort
236,202509,2025-09-01,2025-09,2025,3,9,September,2021-2025
237,202510,2025-10-01,2025-10,2025,4,10,October,2021-2025
238,202511,2025-11-01,2025-11,2025,4,11,November,2021-2025
239,202512,2025-12-01,2025-12,2025,4,12,December,2021-2025
105279,202601,2026-01-01,2026-01,2026,1,1,January,Outside project window


Date dimension range:


,min_date_id,max_date_id,row_count
0,201601,202601,121


In [33]:
print("cpi_clean:", cpi_clean.shape)
print("construction_clean:", construction_clean.shape)
print("events_clean:", events_clean.shape)
print("dim_date_clean:", dim_date_clean.shape)

print("\nConstruction frequency values:")
display(construction_clean["frequency"].drop_duplicates().to_frame())

print("\nConstruction date range:")
display(
    construction_clean
    .agg(
        min_date_id=("date_id", "min"),
        max_date_id=("date_id", "max"),
        row_count=("date_id", "count")
    )
)

cpi_clean: (360, 6)
construction_clean: (37, 8)
events_clean: (6, 15)
dim_date_clean: (121, 8)

Construction frequency values:


,frequency
85911,Quarterly



Construction date range:


,date_id
min_date_id,201701
max_date_id,202601
row_count,37


In [34]:
import sqlite3
from pathlib import Path

project_root = Path("C:/Projects/AnalyticsPortfolio")
db_path = project_root / "data" / "database" / "alberta_home_insurance.db"

conn = sqlite3.connect(db_path)

try:
    # Load dimension table first because fact tables depend on date_id
    dim_date_clean.to_sql(
        "dim_date",
        conn,
        if_exists="append",
        index=False
    )

    cpi_clean.to_sql(
        "fact_cpi",
        conn,
        if_exists="append",
        index=False
    )

    construction_clean.to_sql(
        "fact_construction_cost",
        conn,
        if_exists="append",
        index=False
    )

    events_clean.to_sql(
        "fact_hail_loss_event",
        conn,
        if_exists="append",
        index=False
    )

    conn.commit()
    print("Cleaned data loaded into SQLite successfully.")

finally:
    conn.close()

Cleaned data loaded into SQLite successfully.


In [35]:
import sqlite3
import pandas as pd
from pathlib import Path

project_root = Path("C:/Projects/AnalyticsPortfolio")
db_path = project_root / "data" / "database" / "alberta_home_insurance.db"

conn = sqlite3.connect(db_path)

row_counts = pd.read_sql_query(
    """
    SELECT 'dim_date' AS table_name, COUNT(*) AS row_count FROM dim_date
    UNION ALL
    SELECT 'fact_cpi', COUNT(*) FROM fact_cpi
    UNION ALL
    SELECT 'fact_construction_cost', COUNT(*) FROM fact_construction_cost
    UNION ALL
    SELECT 'fact_hail_loss_event', COUNT(*) FROM fact_hail_loss_event;
    """,
    conn
)

conn.close()

row_counts

,table_name,row_count
0,dim_date,121
1,fact_cpi,360
2,fact_construction_cost,37
3,fact_hail_loss_event,6


In [36]:
# Milestone 2.6 – Data Quality Checks
import sqlite3
import pandas as pd
from pathlib import Path

project_root = Path("C:/Projects/AnalyticsPortfolio")
db_path = project_root / "data" / "database" / "alberta_home_insurance.db"

conn = sqlite3.connect(db_path)

row_counts = pd.read_sql_query(
    """
    SELECT 'dim_date' AS table_name, COUNT(*) AS row_count FROM dim_date
    UNION ALL
    SELECT 'fact_cpi', COUNT(*) FROM fact_cpi
    UNION ALL
    SELECT 'fact_construction_cost', COUNT(*) FROM fact_construction_cost
    UNION ALL
    SELECT 'fact_hail_loss_event', COUNT(*) FROM fact_hail_loss_event;
    """,
    conn
)

conn.close()

row_counts

,table_name,row_count
0,dim_date,121
1,fact_cpi,360
2,fact_construction_cost,37
3,fact_hail_loss_event,6


In [37]:
conn = sqlite3.connect(db_path)

pk_checks = pd.read_sql_query(
    """
    SELECT
        'dim_date' AS table_name,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT date_id) AS distinct_key_count,
        COUNT(*) - COUNT(DISTINCT date_id) AS duplicate_key_count
    FROM dim_date

    UNION ALL

    SELECT
        'fact_cpi',
        COUNT(*),
        COUNT(DISTINCT cpi_id),
        COUNT(*) - COUNT(DISTINCT cpi_id)
    FROM fact_cpi

    UNION ALL

    SELECT
        'fact_construction_cost',
        COUNT(*),
        COUNT(DISTINCT construction_id),
        COUNT(*) - COUNT(DISTINCT construction_id)
    FROM fact_construction_cost

    UNION ALL

    SELECT
        'fact_hail_loss_event',
        COUNT(*),
        COUNT(DISTINCT event_id),
        COUNT(*) - COUNT(DISTINCT event_id)
    FROM fact_hail_loss_event;
    """,
    conn
)

conn.close()

pk_checks

,table_name,total_rows,distinct_key_count,duplicate_key_count
0,dim_date,121,121,0
1,fact_cpi,360,360,0
2,fact_construction_cost,37,37,0
3,fact_hail_loss_event,6,6,0


In [38]:
conn = sqlite3.connect(db_path)

null_checks = pd.read_sql_query(
    """
    SELECT 'dim_date.date_id' AS field_name, COUNT(*) AS null_count FROM dim_date WHERE date_id IS NULL
    UNION ALL
    SELECT 'dim_date.calendar_date', COUNT(*) FROM dim_date WHERE calendar_date IS NULL
    UNION ALL
    SELECT 'dim_date.year_month', COUNT(*) FROM dim_date WHERE year_month IS NULL

    UNION ALL
    SELECT 'fact_cpi.date_id', COUNT(*) FROM fact_cpi WHERE date_id IS NULL
    UNION ALL
    SELECT 'fact_cpi.geography', COUNT(*) FROM fact_cpi WHERE geography IS NULL
    UNION ALL
    SELECT 'fact_cpi.cpi_category', COUNT(*) FROM fact_cpi WHERE cpi_category IS NULL
    UNION ALL
    SELECT 'fact_cpi.cpi_value', COUNT(*) FROM fact_cpi WHERE cpi_value IS NULL

    UNION ALL
    SELECT 'fact_construction_cost.date_id', COUNT(*) FROM fact_construction_cost WHERE date_id IS NULL
    UNION ALL
    SELECT 'fact_construction_cost.geography', COUNT(*) FROM fact_construction_cost WHERE geography IS NULL
    UNION ALL
    SELECT 'fact_construction_cost.construction_category', COUNT(*) FROM fact_construction_cost WHERE construction_category IS NULL
    UNION ALL
    SELECT 'fact_construction_cost.construction_index_value', COUNT(*) FROM fact_construction_cost WHERE construction_index_value IS NULL

    UNION ALL
    SELECT 'fact_hail_loss_event.event_date', COUNT(*) FROM fact_hail_loss_event WHERE event_date IS NULL
    UNION ALL
    SELECT 'fact_hail_loss_event.event_name', COUNT(*) FROM fact_hail_loss_event WHERE event_name IS NULL
    UNION ALL
    SELECT 'fact_hail_loss_event.revised_loss_cad', COUNT(*) FROM fact_hail_loss_event WHERE revised_loss_cad IS NULL;
    """,
    conn
)

conn.close()

null_checks

,field_name,null_count
0,dim_date.date_id,0
1,dim_date.calendar_date,0
2,dim_date.year_month,0
3,fact_cpi.date_id,0
4,fact_cpi.geography,0
5,fact_cpi.cpi_category,0
6,fact_cpi.cpi_value,0
7,fact_construction_cost.date_id,0
8,fact_construction_cost.geography,0
9,fact_construction_cost.construction_category,0


In [39]:
conn = sqlite3.connect(db_path)

date_join_checks = pd.read_sql_query(
    """
    SELECT
        'fact_cpi' AS table_name,
        COUNT(*) AS orphan_date_id_count
    FROM fact_cpi f
    LEFT JOIN dim_date d
        ON f.date_id = d.date_id
    WHERE d.date_id IS NULL

    UNION ALL

    SELECT
        'fact_construction_cost',
        COUNT(*)
    FROM fact_construction_cost f
    LEFT JOIN dim_date d
        ON f.date_id = d.date_id
    WHERE d.date_id IS NULL

    UNION ALL

    SELECT
        'fact_hail_loss_event',
        COUNT(*)
    FROM fact_hail_loss_event f
    LEFT JOIN dim_date d
        ON f.date_id = d.date_id
    WHERE d.date_id IS NULL;
    """,
    conn
)

conn.close()

date_join_checks

,table_name,orphan_date_id_count
0,fact_cpi,0
1,fact_construction_cost,0
2,fact_hail_loss_event,0


In [40]:
conn = sqlite3.connect(db_path)

date_coverage_checks = pd.read_sql_query(
    """
    SELECT
        'dim_date' AS table_name,
        MIN(date_id) AS min_date_id,
        MAX(date_id) AS max_date_id,
        COUNT(*) AS row_count
    FROM dim_date

    UNION ALL

    SELECT
        'fact_cpi',
        MIN(date_id),
        MAX(date_id),
        COUNT(*)
    FROM fact_cpi

    UNION ALL

    SELECT
        'fact_construction_cost',
        MIN(date_id),
        MAX(date_id),
        COUNT(*)
    FROM fact_construction_cost

    UNION ALL

    SELECT
        'fact_hail_loss_event',
        MIN(date_id),
        MAX(date_id),
        COUNT(*)
    FROM fact_hail_loss_event;
    """,
    conn
)

conn.close()

date_coverage_checks

,table_name,min_date_id,max_date_id,row_count
0,dim_date,201601,202601,121
1,fact_cpi,201601,202512,360
2,fact_construction_cost,201701,202601,37
3,fact_hail_loss_event,202006,202508,6


In [41]:
conn = sqlite3.connect(db_path)

value_checks = pd.read_sql_query(
    """
    SELECT
        'fact_cpi.cpi_value <= 0' AS check_name,
        COUNT(*) AS issue_count
    FROM fact_cpi
    WHERE cpi_value <= 0

    UNION ALL

    SELECT
        'fact_construction_cost.construction_index_value <= 0',
        COUNT(*)
    FROM fact_construction_cost
    WHERE construction_index_value <= 0

    UNION ALL

    SELECT
        'fact_hail_loss_event.revised_loss_cad <= 0',
        COUNT(*)
    FROM fact_hail_loss_event
    WHERE revised_loss_cad <= 0

    UNION ALL

    SELECT
        'fact_hail_loss_event.revision_flag not 0/1',
        COUNT(*)
    FROM fact_hail_loss_event
    WHERE revision_flag NOT IN (0, 1);
    """,
    conn
)

conn.close()

value_checks

,check_name,issue_count
0,fact_cpi.cpi_value <= 0,0
1,fact_construction_cost.construction_index_valu...,0
2,fact_hail_loss_event.revised_loss_cad <= 0,0
3,fact_hail_loss_event.revision_flag not 0/1,0


In [42]:
conn = sqlite3.connect(db_path)

qa_summary = pd.DataFrame({
    "qa_check": [
        "Row counts match expected loaded tables",
        "Primary keys are unique",
        "Required fields have no missing values",
        "Fact table date_ids join successfully to dim_date",
        "Date coverage is reasonable for project scope",
        "Numeric values are positive and valid",
        "Revision flags are valid 0/1 values"
    ],
    "status": [
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS"
    ],
    "notes": [
        "dim_date=121, fact_cpi=360, fact_construction_cost=37, fact_hail_loss_event=6",
        "No duplicate primary keys found",
        "No nulls found in required fields",
        "No orphan date_id values found",
        "CPI covers 2016-01 to 2025-12; construction covers 2017-01 to 2026-01; events cover selected curated events from 2020 to 2025",
        "No zero or negative CPI, construction index, or revised loss values found",
        "revision_flag values are valid"
    ]
})

conn.close()

qa_summary


,qa_check,status,notes
0,Row counts match expected loaded tables,PASS,"dim_date=121, fact_cpi=360, fact_construction_..."
1,Primary keys are unique,PASS,No duplicate primary keys found
2,Required fields have no missing values,PASS,No nulls found in required fields
3,Fact table date_ids join successfully to dim_date,PASS,No orphan date_id values found
4,Date coverage is reasonable for project scope,PASS,CPI covers 2016-01 to 2025-12; construction co...
5,Numeric values are positive and valid,PASS,"No zero or negative CPI, construction index, o..."
6,Revision flags are valid 0/1 values,PASS,revision_flag values are valid


## Milestone 2.6 Outcome

Data quality checks passed.

Completed checks:

- Row count verification
- Primary key uniqueness checks
- Required field null checks
- Fact-to-date join checks
- Date coverage review
- Invalid numeric value checks
- Revision flag validation

Known limitation:

The catastrophe event table is curated rather than exhaustive. It contains selected major Alberta insured-loss events used as contextual markers in the analysis. It should not be interpreted as a complete catastrophe-loss database.

Status:

Milestone 2.6 approved.

In [43]:
import sqlite3
from pathlib import Path

project_root = Path("C:/Projects/AnalyticsPortfolio")

db_path = project_root / "data" / "database" / "alberta_home_insurance.db"
views_script_path = project_root / "sql" / "04_create_views.sql"

print("Database path:")
print(db_path)

print("\nViews script path:")
print(views_script_path)

print("\nDatabase exists:", db_path.exists())
print("Views script exists:", views_script_path.exists())

Database path:
C:\Projects\AnalyticsPortfolio\data\database\alberta_home_insurance.db

Views script path:
C:\Projects\AnalyticsPortfolio\sql\04_create_views.sql

Database exists: True
Views script exists: True


In [44]:
views_script = views_script_path.read_text(encoding="utf-8")

conn = sqlite3.connect(db_path)

try:
    conn.executescript(views_script)
    conn.commit()
    print("SQL views created successfully.")
finally:
    conn.close()

SQL views created successfully.


In [45]:
import sqlite3
import pandas as pd
from pathlib import Path

project_root = Path("C:/Projects/AnalyticsPortfolio")
db_path = project_root / "data" / "database" / "alberta_home_insurance.db"

conn = sqlite3.connect(db_path)

views = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'view'
    ORDER BY name;
    """,
    conn
)

conn.close()

views

,name
0,vw_annual_summary
1,vw_catastrophe_events
2,vw_construction_quarterly
3,vw_cpi_monthly


In [46]:
conn = sqlite3.connect(db_path)

view_counts = pd.read_sql_query(
    """
    SELECT 'vw_cpi_monthly' AS view_name, COUNT(*) AS row_count FROM vw_cpi_monthly
    UNION ALL
    SELECT 'vw_construction_quarterly', COUNT(*) FROM vw_construction_quarterly
    UNION ALL
    SELECT 'vw_catastrophe_events', COUNT(*) FROM vw_catastrophe_events
    UNION ALL
    SELECT 'vw_annual_summary', COUNT(*) FROM vw_annual_summary;
    """,
    conn
)

conn.close()

view_counts

,view_name,row_count
0,vw_cpi_monthly,360
1,vw_construction_quarterly,37
2,vw_catastrophe_events,6
3,vw_annual_summary,10


In [47]:
conn = sqlite3.connect(db_path)

annual_summary_preview = pd.read_sql_query(
    """
    SELECT *
    FROM vw_annual_summary
    ORDER BY year;
    """,
    conn
)

conn.close()

annual_summary_preview

,year,five_year_cohort,alberta_all_items_cpi_avg,alberta_home_insurance_cpi_avg,canada_home_insurance_cpi_avg,alberta_residential_construction_index_avg,catastrophe_event_count,total_revised_catastrophe_loss_cad
0,2016,2016-2020,135.166667,356.075000,219.066667,NaN,0,NaN
1,2017,2016-2020,137.250000,364.733333,225.808333,58.050,0,NaN
2,2018,2016-2020,140.633333,372.041667,234.358333,60.750,0,NaN
3,2019,2016-2020,143.066667,384.333333,248.150000,61.675,0,NaN
4,2020,2016-2020,144.658333,409.658333,262.008333,64.100,1,1.300000e+09
5,2021,2021-2025,149.266667,439.133333,275.266667,80.425,1,7.000000e+08
6,2022,2021-2025,158.925000,471.191667,295.800000,97.050,0,NaN
7,2023,2021-2025,164.141667,509.816667,318.250000,100.000,1,1.100000e+08
8,2024,2021-2025,168.908333,562.241667,342.900000,105.650,1,3.293000e+09
9,2025,2021-2025,172.191667,621.875000,360.341667,110.425,2,3.990000e+08


In [48]:
conn = sqlite3.connect(db_path)

view_sql = pd.read_sql_query(
    """
    SELECT sql
    FROM sqlite_master
    WHERE type = 'view'
      AND name = 'vw_annual_summary';
    """,
    conn
)

conn.close()

print(view_sql.loc[0, "sql"])

CREATE VIEW vw_annual_summary AS
WITH cpi_annual AS (
    SELECT
        d.year,
        d.five_year_cohort,

        AVG(CASE
            WHEN c.geography = 'Alberta'
             AND c.cpi_category = 'All-items'
            THEN c.cpi_value
        END) AS alberta_all_items_cpi_avg,

        AVG(CASE
            WHEN c.geography = 'Alberta'
             AND c.cpi_category = 'Homeowners'' home and mortgage insurance'
            THEN c.cpi_value
        END) AS alberta_home_insurance_cpi_avg,

        AVG(CASE
            WHEN c.geography = 'Canada'
             AND c.cpi_category = 'Homeowners'' home and mortgage insurance'
            THEN c.cpi_value
        END) AS canada_home_insurance_cpi_avg

    FROM dim_date d
    LEFT JOIN fact_cpi c
        ON d.date_id = c.date_id
    WHERE d.year BETWEEN 2016 AND 2025
    GROUP BY
        d.year,
        d.five_year_cohort
),

construction_annual AS (
    SELECT
        d.year,
        AVG(k.construction_index_value) AS alberta_residentia

In [49]:
conn = sqlite3.connect(db_path)

annual_summary_fix_sql = """
DROP VIEW IF EXISTS vw_annual_summary;

CREATE VIEW vw_annual_summary AS
WITH cpi_annual AS (
    SELECT
        d.year,
        d.five_year_cohort,

        AVG(CASE
            WHEN c.geography = 'Alberta'
             AND c.cpi_category = 'All-items'
            THEN c.cpi_value
        END) AS alberta_all_items_cpi_avg,

        AVG(CASE
            WHEN c.geography = 'Alberta'
             AND c.cpi_category = 'Homeowners'' home and mortgage insurance'
            THEN c.cpi_value
        END) AS alberta_home_insurance_cpi_avg,

        AVG(CASE
            WHEN c.geography = 'Canada'
             AND c.cpi_category = 'Homeowners'' home and mortgage insurance'
            THEN c.cpi_value
        END) AS canada_home_insurance_cpi_avg

    FROM dim_date d
    LEFT JOIN fact_cpi c
        ON d.date_id = c.date_id
    WHERE d.year BETWEEN 2016 AND 2025
    GROUP BY
        d.year,
        d.five_year_cohort
),

construction_annual AS (
    SELECT
        d.year,
        AVG(k.construction_index_value) AS alberta_residential_construction_index_avg
    FROM fact_construction_cost k
    INNER JOIN dim_date d
        ON k.date_id = d.date_id
    WHERE d.year BETWEEN 2016 AND 2025
    GROUP BY
        d.year
),

events_annual AS (
    SELECT
        d.year,
        COUNT(DISTINCT e.event_id) AS catastrophe_event_count,
        SUM(e.revised_loss_cad) AS total_revised_catastrophe_loss_cad
    FROM fact_hail_loss_event e
    INNER JOIN dim_date d
        ON e.date_id = d.date_id
    WHERE d.year BETWEEN 2016 AND 2025
    GROUP BY
        d.year
)

SELECT
    c.year,
    c.five_year_cohort,
    c.alberta_all_items_cpi_avg,
    c.alberta_home_insurance_cpi_avg,
    c.canada_home_insurance_cpi_avg,
    k.alberta_residential_construction_index_avg,
    COALESCE(e.catastrophe_event_count, 0) AS catastrophe_event_count,
    e.total_revised_catastrophe_loss_cad
FROM cpi_annual c
LEFT JOIN construction_annual k
    ON c.year = k.year
LEFT JOIN events_annual e
    ON c.year = e.year
ORDER BY
    c.year;
"""

conn.executescript(annual_summary_fix_sql)
conn.commit()
conn.close()

print("vw_annual_summary replaced successfully.")

vw_annual_summary replaced successfully.


In [50]:
conn = sqlite3.connect(db_path)

annual_summary_preview = pd.read_sql_query(
    """
    SELECT *
    FROM vw_annual_summary
    ORDER BY year;
    """,
    conn
)

conn.close()

annual_summary_preview

,year,five_year_cohort,alberta_all_items_cpi_avg,alberta_home_insurance_cpi_avg,canada_home_insurance_cpi_avg,alberta_residential_construction_index_avg,catastrophe_event_count,total_revised_catastrophe_loss_cad
0,2016,2016-2020,135.166667,356.075000,219.066667,NaN,0,NaN
1,2017,2016-2020,137.250000,364.733333,225.808333,58.050,0,NaN
2,2018,2016-2020,140.633333,372.041667,234.358333,60.750,0,NaN
3,2019,2016-2020,143.066667,384.333333,248.150000,61.675,0,NaN
4,2020,2016-2020,144.658333,409.658333,262.008333,64.100,1,1.300000e+09
5,2021,2021-2025,149.266667,439.133333,275.266667,80.425,1,7.000000e+08
6,2022,2021-2025,158.925000,471.191667,295.800000,97.050,0,NaN
7,2023,2021-2025,164.141667,509.816667,318.250000,100.000,1,1.100000e+08
8,2024,2021-2025,168.908333,562.241667,342.900000,105.650,1,3.293000e+09
9,2025,2021-2025,172.191667,621.875000,360.341667,110.425,2,3.990000e+08


In [51]:
conn = sqlite3.connect(db_path)

view_qa = pd.read_sql_query(
    """
    SELECT
        'vw_cpi_monthly' AS view_name,
        COUNT(*) AS row_count,
        MIN(date_id) AS min_date_id,
        MAX(date_id) AS max_date_id
    FROM vw_cpi_monthly

    UNION ALL

    SELECT
        'vw_construction_quarterly',
        COUNT(*),
        MIN(date_id),
        MAX(date_id)
    FROM vw_construction_quarterly

    UNION ALL

    SELECT
        'vw_catastrophe_events',
        COUNT(*),
        MIN(date_id),
        MAX(date_id)
    FROM vw_catastrophe_events

    UNION ALL

    SELECT
        'vw_annual_summary',
        COUNT(*),
        MIN(year),
        MAX(year)
    FROM vw_annual_summary;
    """,
    conn
)

conn.close()

view_qa

,view_name,row_count,min_date_id,max_date_id
0,vw_cpi_monthly,360,201601,202512
1,vw_construction_quarterly,37,201701,202601
2,vw_catastrophe_events,6,202006,202508
3,vw_annual_summary,10,2016,2025


## Milestone 2.7 Outcome

Cleaning and transformation views were created successfully.

Created views:

- `vw_cpi_monthly`
- `vw_construction_quarterly`
- `vw_catastrophe_events`
- `vw_annual_summary`

A join-grain issue was detected in the first version of `vw_annual_summary`, where catastrophe losses were multiplied by the number of CPI series joined at the monthly level.

The issue was corrected by pre-aggregating CPI, construction, and catastrophe event data separately before joining annual results.

Status:

Milestone 2.7 approved.

# Milestone 2.8 – Validation

In [52]:
conn = sqlite3.connect(db_path)

event_loss_validation = pd.read_sql_query(
    """
    SELECT
        d.year,
        COUNT(e.event_id) AS event_count,
        SUM(e.revised_loss_cad) AS total_event_loss
    FROM fact_hail_loss_event e
    INNER JOIN dim_date d
        ON e.date_id = d.date_id
    GROUP BY d.year
    ORDER BY d.year;
    """,
    conn
)

annual_summary_event_check = pd.read_sql_query(
    """
    SELECT
        year,
        catastrophe_event_count,
        total_revised_catastrophe_loss_cad
    FROM vw_annual_summary
    WHERE catastrophe_event_count > 0
    ORDER BY year;
    """,
    conn
)

conn.close()

print("Event totals from base fact table:")
display(event_loss_validation)

print("Event totals from annual summary view:")
display(annual_summary_event_check)

Event totals from base fact table:


,year,event_count,total_event_loss
0,2020,1,1.300000e+09
1,2021,1,7.000000e+08
2,2023,1,1.100000e+08
3,2024,1,3.293000e+09
4,2025,2,3.990000e+08


Event totals from annual summary view:


,year,catastrophe_event_count,total_revised_catastrophe_loss_cad
0,2020,1,1.300000e+09
1,2021,1,7.000000e+08
2,2023,1,1.100000e+08
3,2024,1,3.293000e+09
4,2025,2,3.990000e+08


In [53]:
conn = sqlite3.connect(db_path)

cpi_series_validation = pd.read_sql_query(
    """
    SELECT
        geography,
        cpi_category,
        COUNT(*) AS month_count,
        MIN(date_id) AS min_date_id,
        MAX(date_id) AS max_date_id,
        MIN(cpi_value) AS min_cpi_value,
        MAX(cpi_value) AS max_cpi_value
    FROM vw_cpi_monthly
    GROUP BY
        geography,
        cpi_category
    ORDER BY
        geography,
        cpi_category;
    """,
    conn
)

conn.close()

cpi_series_validation

,geography,cpi_category,month_count,min_date_id,max_date_id,min_cpi_value,max_cpi_value
0,Alberta,All-items,120,201601,202512,133.7,173.4
1,Alberta,Homeowners' home and mortgage insurance,120,201601,202512,350.0,647.4
2,Canada,Homeowners' home and mortgage insurance,120,201601,202512,215.8,368.5


In [54]:
conn = sqlite3.connect(db_path)

construction_validation = pd.read_sql_query(
    """
    SELECT
        geography,
        construction_category,
        frequency,
        geographic_proxy_flag,
        COUNT(*) AS period_count,
        MIN(date_id) AS min_date_id,
        MAX(date_id) AS max_date_id,
        MIN(construction_index_value) AS min_index_value,
        MAX(construction_index_value) AS max_index_value
    FROM vw_construction_quarterly
    GROUP BY
        geography,
        construction_category,
        frequency,
        geographic_proxy_flag;
    """,
    conn
)

conn.close()

construction_validation

,geography,construction_category,frequency,geographic_proxy_flag,period_count,min_date_id,max_date_id,min_index_value,max_index_value
0,Alberta,Residential buildings [621],Quarterly,Province-wide,37,201701,202601,56.9,111.0
